# CSBS BS REDCap Validation (PID 6207)

This notebook verifies that REDCap calculated fields for the CSBS BS instrument match the approved worksheet logic.
It is validation-only and should not be used to derive new logic.

In [ ]:
import os
import requests
from decimal import Decimal, ROUND_HALF_UP

API_URL = os.getenv('REDCAP_API_URL', 'https://redcap.research.sc.edu/api/')
TOKEN = os.getenv('NANODD_API_TOKEN', '').strip()
if not TOKEN:
    raise RuntimeError('Set NANODD_API_TOKEN in your environment before running this notebook.')

def post(content, **params):
    data = {'token': TOKEN, 'content': content, 'format': 'json', 'returnFormat': 'json'}
    data.update(params)
    r = requests.post(API_URL, data=data, timeout=180)
    r.raise_for_status()
    payload = r.json()
    if isinstance(payload, dict) and payload.get('error'):
        raise RuntimeError(payload['error'])
    return payload

project = post('project')
project = project[0] if isinstance(project, list) and project else project
print('Project ID:', project.get('project_id'))
print('Project Title:', project.get('project_title'))
if str(project.get('project_id')) != '6207':
    raise RuntimeError('Token does not map to PID 6207.')

In [ ]:
def num(v):
    if v in (None, ''):
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def half_up(v):
    return float(Decimal(str(v)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def bs_expected(row):
    required = [f'csbsbs_scale{i}' for i in range(1, 16)] + [
        'csbsbs_scale16_1', 'csbsbs_scale16_2', 'csbsbs_scale16_3',
        'csbsbs_scale17', 'csbsbs_scale18', 'csbsbs_scale19', 'csbsbs_scale20'
    ]
    if any(num(row.get(k)) is None for k in required):
        return None

    g = lambda name: num(row.get(name))
    emotion = half_up(g('csbsbs_scale1') + g('csbsbs_scale2') + 3 * g('csbsbs_scale3'))
    communication = half_up(g('csbsbs_scale4') / 3 + g('csbsbs_scale5') + g('csbsbs_scale6') + g('csbsbs_scale7'))
    gestures = half_up(2 * g('csbsbs_scale8') + g('csbsbs_scale9'))
    sounds = half_up(g('csbsbs_scale10') + 2 * g('csbsbs_scale11'))
    words = half_up(g('csbsbs_scale12') + g('csbsbs_scale13') / 2 + g('csbsbs_scale14') + g('csbsbs_scale15'))
    understanding = half_up(3 * (g('csbsbs_scale16_1') + g('csbsbs_scale16_2') + g('csbsbs_scale16_3')))
    object_use = half_up(g('csbsbs_scale17') + g('csbsbs_scale18') + g('csbsbs_scale19') + g('csbsbs_scale20'))

    return {
        'csbsbs_emotionraw': emotion,
        'csbsbs_comraw': communication,
        'csbsbs_gesraw': gestures,
        'csbsbs_soundsraw': sounds,
        'csbsbs_wordsraw': words,
        'csbsbs_underraw': understanding,
        'csbsbs_objectraw': object_use,
        'csbsbs_socialcompositecalc': emotion + communication + gestures,
        'csbsbs_speechcompositecalc': sounds + words,
        'csbsbs_symboliccompositecalc': understanding + object_use,
        'csbsbs_totalrawcalc': emotion + communication + gestures + sounds + words + understanding + object_use,
    }

In [ ]:
metadata = post('metadata')
record_id_field = metadata[0]['field_name']
score_fields = [
    'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
    'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
    'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
]
input_fields = [f'csbsbs_scale{i}' for i in range(1,16)] + [
    'csbsbs_scale16_1','csbsbs_scale16_2','csbsbs_scale16_3','csbsbs_scale17','csbsbs_scale18','csbsbs_scale19','csbsbs_scale20'
]
fields = [record_id_field, 'redcap_event_name', 'csbs_bs_complete'] + input_fields + score_fields
records = post('record', type='flat', forms=['csbs_bs'], fields=fields, rawOrLabel='raw', rawOrLabelHeaders='raw')

comparisons = 0
mismatches = []
for row in records:
    exp = bs_expected(row)
    if exp is None:
        continue
    rid = row.get(record_id_field, '')
    ev = row.get('redcap_event_name', '')
    for f, expected in exp.items():
        actual = num(row.get(f))
        if actual is None:
            mismatches.append((rid, ev, f, 'missing', expected))
            continue
        comparisons += 1
        if abs(actual - expected) > 1e-9:
            mismatches.append((rid, ev, f, actual, expected))

print('Record ID field:', record_id_field)
print('Records exported:', len(records))
print('Score comparisons:', comparisons)
print('Mismatches:', len(mismatches))
if mismatches:
    print('First 20 mismatches:')
    for row in mismatches[:20]:
        print(row)

if len(mismatches) == 0:
    print('Validation status: PASS (REDCap matches approved CSBS BS scoring logic)')
else:
    print('Validation status: REVIEW REQUIRED')